In [2]:
# import libraries
import polars as pl
import plotly.express as px
from pathlib import Path

#make path and load datasets
file_path = Path("ecommerce_sales_data.csv")
df = pl.read_csv(file_path)
print(df.head())
print(df.describe())
print(df.null_count())

# convert from str  to date format
df = df.with_columns(
    pl.col("Order Date").str.strptime(pl.Date, "%Y-%m-%d", strict=False)
)

# upload cleaner dataset version
df.write_csv("cleaned_ecommerce_sales_data.csv")

# aggregation
df_category = (
    df.group_by("Category")
       .agg([
        pl.sum("Sales").alias("total_sales"),
        pl.sum("Profit").alias("total_profit")
    ]).sort("total_sales", descending=True)
)

df_product = (
    df.group_by("Product Name")
     .agg([
        pl.sum("Sales").alias("total_sales"),
        pl.sum("Profit").alias("total_profit"),
    ])
    .sort("total_sales", descending=True).head(10)
)

df_region = (
    df.group_by("Region", maintain_order=True)
       .agg([
        pl.sum("Sales").alias("total_sales"),
        pl.sum("Profit").alias("total_profit"),
        pl.sum("Quantity").alias("total_quantity")
    ])
)

#conver to pandas for plotly
category_pd = df_category.to_pandas()
product_pd = df_product.to_pandas()
region_pd = df_region.to_pandas()

# visualization
fig1 = px.bar(
    category_pd,
    x="Category",

    y="total_sales",
    color="Category",
    hover_data=["total_profit"],
    text_auto=".3s",
    title="Total Sales by Category"
)
fig1.show()

fig2 = px.bar(
    product_pd,
    x="Product Name",
    y="total_sales",
    color="Product Name",
    text_auto=".3s",
    hover_data=["total_profit"],
    title="Total Sales by Product"
)
fig2.show()

fig3 = px.scatter(
    region_pd,
    x="Region",
    y="total_sales",
    size="total_quantity",
    color="Region",
    hover_data=["total_profit"],
    trendline="ols",
    title="Total Sales by Region"
)
fig3.show()


shape: (5, 7)
┌────────────┬──────────────┬─────────────┬────────┬──────────┬───────┬────────┐
│ Order Date ┆ Product Name ┆ Category    ┆ Region ┆ Quantity ┆ Sales ┆ Profit │
│ ---        ┆ ---          ┆ ---         ┆ ---    ┆ ---      ┆ ---   ┆ ---    │
│ str        ┆ str          ┆ str         ┆ str    ┆ i64      ┆ i64   ┆ f64    │
╞════════════╪══════════════╪═════════════╪════════╪══════════╪═══════╪════════╡
│ 2024-12-31 ┆ Printer      ┆ Office      ┆ North  ┆ 4        ┆ 3640  ┆ 348.93 │
│ 2022-11-27 ┆ Mouse        ┆ Accessories ┆ East   ┆ 7        ┆ 1197  ┆ 106.53 │
│ 2022-05-11 ┆ Tablet       ┆ Electronics ┆ South  ┆ 5        ┆ 5865  ┆ 502.73 │
│ 2024-03-16 ┆ Mouse        ┆ Accessories ┆ South  ┆ 2        ┆ 786   ┆ 202.87 │
│ 2022-09-10 ┆ Mouse        ┆ Accessories ┆ West   ┆ 1        ┆ 509   ┆ 103.28 │
└────────────┴──────────────┴─────────────┴────────┴──────────┴───────┴────────┘
shape: (9, 8)
┌────────────┬────────────┬─────────┬──────────────┬────────┬──────────┬─────────